# PathPilot — Review Aspect-Based Sentiment Analysis (RoBERTa)

**Owner:** Anayan (`feature/nlp-model`)  
**Project:** PathPilot (Kolkata-first AI-assisted travel-safety Android prototype)

## Epistemic & Scope Notes
- **SemEval-2014 Task 4 (Restaurants)** is used here strictly as an official **benchmark/transfer** dataset; restaurant reviews are **not** Kolkata travel-safety ground truth.
- Local Kolkata review snippets (when collected via permitted sources or consented contributions) must follow `docs/nlp/annotation_guide.md`, `templates/review_annotation_template.csv`, and PII anonymisation before training.
- Review NLP outputs **perceptions** (`aspect`, `sentiment`, `confidence`, `evidence_text`, `source_date`, `model_version`, `limitations`) and **never** computes a route safety score.

## 1. Setup, Versions, Deterministic Seeds & Configurable Drive Path

In [ ]:
import os, sys, json, random, hashlib, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Install required packages in fresh Google Colab runtime
    os.system("pip -q install transformers datasets scikit-learn pandas numpy accelerate defusedxml huggingface_hub matplotlib")

import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configurable Google Drive / local base path
USE_DRIVE = False
DRIVE_BASE_DIR = "/content/drive/MyDrive/PathPilot/nlp"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path(DRIVE_BASE_DIR)
else:
    BASE_DIR = Path.cwd()
    # If running from ml/nlp/training/, step up to repo root
    if BASE_DIR.name == "training" and (BASE_DIR.parent.parent.parent / "ml").exists():
        BASE_DIR = BASE_DIR.parent.parent.parent

RAW_DIR = BASE_DIR / "data" / "raw" / "nlp" / "semeval2014"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nlp"
OUTPUT_DIR = BASE_DIR / "artifacts" / "models" / "pathpilot-roberta-absa"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"Transformers: {transformers.__version__} | Scikit-learn: {sklearn.__version__}")
print(f"BASE_DIR: {BASE_DIR}")

## 2. Dataset Acquisition, Schema Validation, Privacy Cleaning & Deduplication

In [ ]:
from ml.nlp.labels import PATHPILOT_ASPECTS, POLARITIES, SEMEVAL_2014_RESTAURANT_ASPECTS
from ml.nlp.preprocess import (
    anonymise_text,
    assert_no_split_leakage,
    deduplicate_records,
    download_semeval2014_dataset,
    grouped_train_val_split,
    parse_semeval2014_restaurant_xml,
    remove_cross_split_text_leakage,
    validate_annotation_df,
)

# 2a. Verify official SemEval-2014 Task 4 XML files & SHA-256 checksums
resolved_files = download_semeval2014_dataset(RAW_DIR)
for key, (fpath, sha) in resolved_files.items():
    print(f"{key:24s} -> {fpath.name:30s} sha256={sha[:16]}...")

# 2b. Parse & anonymise train and untouched gold test sets
full_train_df, train_stats = parse_semeval2014_restaurant_xml(
    resolved_files["restaurants_train"][0], source_id="semeval2014_rest_train"
)
test_df, test_stats = parse_semeval2014_restaurant_xml(
    resolved_files["restaurants_test_gold"][0], source_id="semeval2014_rest_test"
)
print("Train parse stats:", train_stats)
print("Test parse stats :", test_stats)

# 2c. Schema validation against PathPilot template constraints
train_errors = validate_annotation_df(full_train_df, allowed_aspects=SEMEVAL_2014_RESTAURANT_ASPECTS)
test_errors = validate_annotation_df(test_df, allowed_aspects=SEMEVAL_2014_RESTAURANT_ASPECTS)
assert not train_errors, f"Train schema errors: {train_errors}"
assert not test_errors, f"Test schema errors: {test_errors}"
print("Schema validation passed for train and test.")

# 2d. Deduplication & cross-split leakage cleanup
full_train_df, train_dedup = deduplicate_records(full_train_df)
test_df, test_dedup = deduplicate_records(test_df)
full_train_df, cross_test_removed = remove_cross_split_text_leakage(full_train_df, test_df, threshold=0.90)
print(f"Dedup removed: train={train_dedup}, test={test_dedup} | cross-test leakage removed={cross_test_removed}")

## 3. Label Distribution, Grouped Split & Leakage Assertions

In [ ]:
train_df, val_df = grouped_train_val_split(
    full_train_df, group_col="group_id", label_col="sentiment", val_size=0.15, seed=SEED
)
train_df, cross_val_removed = remove_cross_split_text_leakage(train_df, val_df, threshold=0.90)
print(f"Cross-val near-duplicate rows removed from train: {cross_val_removed}")

# Strict leakage assertions: group_id, exact normalized text, and char-4gram Jaccard >= 0.90
assert_no_split_leakage(
    {"train": train_df, "val": val_df, "test": test_df},
    group_col="group_id",
    text_col="text",
    near_dup_threshold=0.90,
)
print("Leakage assertions PASSED across train / val / test!")

for split_name, sdf in [("Train", train_df), ("Val", val_df), ("Test (Untouched Gold)", test_df)]:
    print(f"\n--- {split_name} (n={len(sdf)}, unique_groups={sdf['group_id'].nunique()}) ---")
    print("Sentiment counts:", sdf["sentiment"].value_counts().to_dict())
    print("Aspect counts   :", sdf["aspect"].value_counts().to_dict())

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(PROCESSED_DIR / "semeval2014_rest_train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "semeval2014_rest_val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "semeval2014_rest_test.csv", index=False)

## 4. RoBERTa Tokenisation & 10-Epoch Training

In [ ]:
from ml.nlp.train import train_and_evaluate

# On GPU (Colab T4), you can set MODEL_NAME='roberta-base' and FREEZE_BOTTOM_LAYERS=0.
# On CPU, 'distilroberta-base' with FREEZE_BOTTOM_LAYERS=5 (top transformer block + classifier) runs 10 epochs cleanly.
MODEL_NAME = "distilroberta-base"
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
MAX_LENGTH = 96
FREEZE_BOTTOM_LAYERS = 0 if torch.cuda.is_available() else 5

summary = train_and_evaluate(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    max_length=MAX_LENGTH,
    seed=SEED,
    use_class_weights=True,
    freeze_bottom_layers=FREEZE_BOTTOM_LAYERS,
)

epoch_df = pd.DataFrame([
    {
        "epoch": r["epoch"],
        "train_loss": r["train_loss"],
        "train_acc": r["train_accuracy"],
        "train_macro_f1": r["train_macro_f1"],
        "val_loss": r["val_loss"],
        "val_acc": r["val_accuracy"],
        "val_macro_f1": r["val_macro_f1"],
        "val_weighted_f1": r["val_weighted_f1"],
        "epoch_s": r["epoch_seconds"],
        "best": r["is_best"],
    }
    for r in summary["epoch_history"]
])
display(epoch_df) if IN_COLAB else print(epoch_df.to_string(index=False))

## 5. Held-Out Test Evaluation, Confusion Matrix & Error Analysis

In [ ]:
import matplotlib.pyplot as plt

tm = summary["test_metrics"]
print(f"Best validation epoch: {summary['best_epoch']} (val_macro_f1={summary['best_val_macro_f1']:.4f})")
print(f"Held-out Test Loss: {summary['test_loss']:.4f}")
print(f"Held-out Test Accuracy: {tm['accuracy']:.4f} | Macro-F1: {tm['macro_f1']:.4f} | Weighted-F1: {tm['weighted_f1']:.4f}")

print("\nPer-Class Test Metrics:")
print(pd.DataFrame(tm["per_class"]).T)

print("\nPer-Aspect Test Metrics:")
print(pd.DataFrame(tm["per_aspect"]).T)

cm = np.array(tm["confusion_matrix"])
labels = tm["confusion_matrix_labels"]
print("\nConfusion Matrix (rows=true, cols=pred):", labels)
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)), labels)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Held-Out Test Confusion Matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

print("\nTop High-Confidence Misclassifications (Error Analysis):")
print(pd.DataFrame(summary["error_analysis_top15"])[["aspect", "true_sentiment", "pred_sentiment", "confidence", "text"]].head(10).to_string(index=False))

## 6. Save/Reload Artefacts, Inference Demo & CPU Smoke Test

In [ ]:
from ml.nlp.inference import ReviewABSAInference

engine = ReviewABSAInference(model_dir=OUTPUT_DIR, model_version=summary["model_version"])
assert engine.is_loaded, "Reloaded model failed to load!"

demo_payload = engine.analyse_reviews(
    reviews=[
        {
            "review_id": "demo_001",
            "text": "Bright streetlights and open pharmacies near the metro exit until late.",
            "source_id": "demo_smoke_input",
            "source_date": "2026-11-25T21:30:00+05:30",
            "candidate_aspects": ["lighting", "commercial_activity"],
        },
        {
            "review_id": "demo_002",
            "text": "The pasta was great, though service took far too long.",
            "source_id": "semeval_domain_example",
            "source_date": "2026-11-25T21:30:00+05:30",
            "candidate_aspects": ["food", "service"],
        },
    ],
    location_id="kmc_ward_063_park_street",
    allow_mock=False,
)
print("CPU smoke test summary:", summary["cpu_smoke_test"])
print(json.dumps(demo_payload, indent=2))